In [1]:
import pandas as pd
print("version: 1.0.1")

version: 1.0.1


In [2]:
import kagglehub

path = kagglehub.dataset_download("saurabhbagchi/books-dataset")

print("Path to dataset files:", path)

c:\Users\Shaked\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\Shaked\.cache\kagglehub\datasets\saurabhbagchi\books-dataset\versions\1


In [ ]:
import pandas as pd

ratings = pd.read_csv(
    path + r"\books_data\Ratings.csv",
    encoding="latin-1",
    sep=";",
    engine="python",
    on_bad_lines="warn"
)

books = pd.read_csv(
    path + r"\books_data\Books.csv",
    encoding="latin-1",
    sep=";",
    engine="python",
    on_bad_lines="warn"
)

user = pd.read_csv(
    path + r"\books_data\Users.csv",
    encoding="latin-1",
    sep=";",
    engine="python",
    on_bad_lines="warn"
)



Skipping line 21689: ';' expected after '"'
Skipping line 45329: ';' expected after '"'
Skipping line 45330: ';' expected after '"'
Skipping line 183262: ';' expected after '"'
Skipping line 210093: ';' expected after '"'
Skipping line 531938: ';' expected after '"'
Skipping line 663237: ';' expected after '"'
Skipping line 824822: ';' expected after '"'
Skipping line 877349: ';' expected after '"'
Skipping line 941679: ';' expected after '"'
Skipping line 1134132: ';' expected after '"'
Skipping line 1145176: ';' expected after '"'
Skipping line 1145177: ';' expected after '"'
Skipping line 1148551: ';' expected after '"'
Skipping line 815: ';' expected after '"'
Skipping line 1679: ';' expected after '"'
Skipping line 2531: ';' expected after '"'
Skipping line 2640: ';' expected after '"'
Skipping line 3236: ';' expected after '"'
Skipping line 3256: ';' expected after '"'
Skipping line 4906: ';' expected after '"'
Skipping line 6452: ';' expected after '"'
Skipping line 8349: ';' ex

In [12]:
print("head of ratings dataset:")
print(ratings.head())
print("head of books dataset:")
print(books.head())
print("head of user dataset:")
print(user.head())

head of ratings dataset:
   User-ID        ISBN  Book-Rating
0   276725  034545104X            0
1   276726  0155061224            5
2   276727  0446520802            0
3   276729  052165615X            3
4   276729  0521795028            6
head of books dataset:
         ISBN                                         Book-Title  \
0  0195153448                                Classical Mythology   
1  0002005018                                       Clara Callan   
2  0060973129                               Decision in Normandy   
3  0374157065  Flu: The Story of the Great Influenza Pandemic...   
4  0393045218                             The Mummies of Urumchi   

            Book-Author  Year-Of-Publication                   Publisher  \
0    Mark P. O. Morford                 2002     Oxford University Press   
1  Richard Bruce Wright                 2001       HarperFlamingo Canada   
2          Carlo D'Este                 1991             HarperPerennial   
3      Gina Bari Kolata

In [ ]:
book_stats = ratings.groupby("ISBN")["Book-Rating"].agg(
    size="size",
    mean="mean"
).reset_index()
book_stats


top_books_with_info = books.merge(
    book_stats,
    on="ISBN",
    how="left"
)



top_books_with_info[["size", "mean"]].isna().sum()
print(top_books_with_info[["size", "mean"]].isna().sum())
print(top_books_with_info[["size", "mean"]].shape)
# There are 1203 out of 271360 books that have no ratings. 

top_books_with_info = top_books_with_info.fillna({"size": 0, "mean": 0})
print(top_books_with_info[["size", "mean"]].isna().sum())
print(top_books_with_info[["size", "mean"]].shape)



size    1203
mean    1203
dtype: int64
(270491, 2)
size    0
mean    0
dtype: int64
(270491, 2)


In [4]:
# -------- checking if can add one hot encoding to the categorical columns --------
object_cols = [col for col in user.columns if user[col].dtype == "object"]
print(object_cols)
object_nunique = list(map(lambda col: user[col].nunique(), object_cols))
d = dict(zip(object_cols, object_nunique))

# Print number of unique entries by column, in ascending order
print(sorted(d.items(), key=lambda x: x[1]))
# -------- can't --------

['Location']
[('Location', 56882)]


In [24]:
top_books_with_info["Book-Author"] = top_books_with_info["Book-Author"].str.capitalize()
top_books_with_info["Book-Title"] = top_books_with_info["Book-Title"].str.capitalize()
top_books_with_info["Publisher"] = top_books_with_info["Publisher"].str.capitalize()
top_books_with_info["Book-Author"] = top_books_with_info["Book-Author"].fillna("Unknown")
top_books_with_info["Author-Count"] = top_books_with_info.groupby("Book-Author")["ISBN"].transform("count")
top_books_with_info[["Book-Author", "Author-Count", "Book-Title"]].sort_values("Author-Count", ascending=False).head(10)

,Book-Author,Author-Count,Book-Title
167995,Agatha christie,635,Le train bleu
5259,Agatha christie,635,The murder of roger ackroyd
175867,Agatha christie,635,Jeux de glaces (le livre de poche)
215395,Agatha christie,635,Ordeal by inocence
75983,Agatha christie,635,Hickory dickory dock
75981,Agatha christie,635,Evil under the sun
75980,Agatha christie,635,Murder in mesopotamia
75979,Agatha christie,635,The murder at hazelmoor
223764,Agatha christie,635,Mrs. mcginty's dead
110219,Agatha christie,635,The secret of chimneys


In [25]:
top_books_with_info["Publication_Count"] = top_books_with_info.groupby("Publisher")["ISBN"].transform("count")
top_books_with_info[["Publisher", "Publication_Count", "Book-Title"]].sort_values("Publication_Count", ascending=False).head(10)

,Publisher,Publication_Count,Book-Title
137339,Harlequin,7532.0,Lover's leap (heartbeat) (harlequin american r...
210780,Harlequin,7532.0,Whole truth (harlequin superromance 280)
210778,Harlequin,7532.0,"Day before dawn (harlequin superromance, no 295)"
186734,Harlequin,7532.0,In their footsteps (harlequin single title)
148569,Harlequin,7532.0,No strings heartbeat (reader's choice)
125632,Harlequin,7532.0,The sex diet (blaze)
49926,Harlequin,7532.0,Long tall texans iii (by request) (by request...
226030,Harlequin,7532.0,One night of love (top author) (harlequin pres...
216194,Harlequin,7532.0,Legacy
168437,Harlequin,7532.0,Too young to love


In [26]:
top_books_with_info["Author-Avg-Rating"] = top_books_with_info.groupby("Book-Author")["mean"].transform("mean") 
top_books_with_info["Author-Reader-Count"] = top_books_with_info.groupby("Book-Author")["size"].transform("sum")
top_books_with_info["Author-Popularity"] = top_books_with_info["Author-Avg-Rating"] * (top_books_with_info["Author-Reader-Count"] / 1000)
top_books_with_info[["Book-Author", "Author-Avg-Rating", "Author-Reader-Count", "Author-Popularity", "Book-Title"]].sort_values("Author-Popularity", ascending=False).drop_duplicates("Book-Author").head(10)

,Book-Author,Author-Avg-Rating,Author-Reader-Count,Author-Popularity,Book-Title
7841,Stephen king,3.903150,10178.0,39.726256,Dolores claiborne
40444,John grisham,3.813768,8818.0,33.629806,Rey de los pleitos
192204,Nora roberts,2.897050,8590.0,24.885660,Remember when
6031,James patterson,2.887803,5845.0,16.879208,The lake house
3908,J. k. rowling,5.561769,2908.0,16.173624,Quidditch through the ages
25825,Mary higgins clark,3.032529,4908.0,14.883652,"Weep no more, my lady"
2444,Michael crichton,3.311353,4277.0,14.162658,Airframe
189884,Dean r. koontz,3.045478,4313.0,13.135145,The eyes of darkness
9970,Janet evanovich,3.909480,3351.0,13.100667,Visions of sugar plums
26098,Anne rice,3.254278,3922.0,12.763279,Violin


In [11]:
features = [
    "Author-Count",
    "Year-Of-Publication",
    "Publication_Count",
    "size",
    "mean"
]
X = top_books_with_info[features]


In [ ]:

from sklearn.model_selection import train_test_split


X_train, y_train, X_test, y_test = train_test_split(X, top_books_with_info[features], test_size=0.2, random_state=42)

KeyError: 'Book-Rating'

In [ ]:
# y is the user ratings for the books, which is the target variable we want to predict.
